# Factory Compliance & Alert Escalation System — End-to-End Test

This notebook clones the repository and exercises every component that is
currently implemented, in order:

1. Clone the repo + install CPU-only dependencies
2. Provide Kaggle credentials and auto-download the video dataset
3. Provide the compliance policy PDF (upload yours, or generate a synthetic one for a smoke test)
4. Build the offline RAG vector store (PDF → chunks → embeddings → FAISS)
5. Run sample retrieval queries against the policy index
6. Run the ABM core (Mesa model: Policy/Vision/Risk/Escalation/Audit agents) over the downloaded clips
7. Print Green-AI-style timing/resource notes for each stage

Vision/Risk/Escalation/Audit agents are currently interface-stable stubs
(see the repo README's status table), so step 6 will run the full agent
chain but detections/severities/escalations will be empty lists until
those modules are filled in — this notebook still verifies the *wiring*
end-to-end, which is the point of this test pass.

**Runtime:** CPU is sufficient and intentional — go to `Runtime > Change
runtime type` and select **CPU** (no GPU needed, consistent with this
project's Green AI constraints).

## 1. Clone the repository

In [ ]:
# Replace with your actual GitHub repo URL once pushed
REPO_URL = "https://github.com/<your-username>/factory-compliance-system.git"  # @param {type:"string"}
REPO_DIR = "/content/factory-compliance-system"

import os
if not os.path.exists(REPO_DIR):
    !git clone "$REPO_URL" "$REPO_DIR"
else:
    print("Repo already cloned, pulling latest...")
    !cd "$REPO_DIR" && git pull

%cd $REPO_DIR
!ls -la

## 2. Install dependencies (CPU-only wheels)

In [ ]:
!pip install -q -r requirements.txt
print("Dependencies installed.")

## 3. Kaggle credentials

Kaggle requires a personal API token for any programmatic dataset access —
this is the **only** manual input the whole system needs. Get yours from
`https://www.kaggle.com/settings -> API -> Create New Token`, which
downloads a `kaggle.json` file. Run the cell below and upload it when
prompted. (You can also paste the username/key directly into the
fallback cell if you prefer not to upload the file.)

In [ ]:
import json, os
from pathlib import Path

try:
    from google.colab import files
    print("Upload your kaggle.json (skip / cancel to use manual entry instead)")
    uploaded = files.upload()
    if uploaded:
        fname = list(uploaded.keys())[0]
        creds = json.loads(Path(fname).read_text())
        os.environ["KAGGLE_USERNAME"] = creds["username"]
        os.environ["KAGGLE_KEY"] = creds["key"]
        print("Kaggle credentials loaded from uploaded file.")
except Exception as e:
    print(f"Skipping upload flow ({e}); use the manual entry cell below.")

In [ ]:
# Manual fallback — only run this if the upload cell above didn't set the
# env vars. Leave blank and re-run to skip auto-download (the system will
# fall back to whatever clips you place in data/raw_clips/ manually).
import os
KAGGLE_USERNAME = ""  # @param {type:"string"}
KAGGLE_KEY = ""  # @param {type:"string"}
if KAGGLE_USERNAME and KAGGLE_KEY:
    os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
    os.environ["KAGGLE_KEY"] = KAGGLE_KEY
    print("Kaggle credentials set manually.")
else:
    print("No manual credentials entered; relying on previous cell (if any).")

## 4. Auto-download the Kaggle video dataset

In [ ]:
import time
t0 = time.time()
!python3 scripts/download_dataset.py
print(f"Dataset stage took {time.time() - t0:.1f}s")
!find data/raw_clips -type f | head -10
!find data/raw_clips -type f | wc -l

## 5. Provide the compliance policy PDF

Upload your real policy PDF to `data/policy/compliance_policy.pdf`. If you
don't have one handy and just want to smoke-test the pipeline, the next
cell generates a small synthetic policy PDF with the right structural
markers (Section refs, WARNING / CRITICAL SAFETY NOTICE callouts) that the
parser looks for.

In [ ]:
try:
    from google.colab import files
    print("Upload compliance_policy.pdf (skip / cancel to generate a synthetic test PDF instead)")
    uploaded = files.upload()
    if uploaded:
        fname = list(uploaded.keys())[0]
        import shutil
        os.makedirs("data/policy", exist_ok=True)
        shutil.move(fname, "data/policy/compliance_policy.pdf")
        print("Saved uploaded PDF to data/policy/compliance_policy.pdf")
except Exception as e:
    print(f"Skipping upload flow ({e})")

In [ ]:
import os
if not os.path.exists("data/policy/compliance_policy.pdf"):
    print("No policy PDF found — generating a synthetic test document for the smoke test.")
    !pip install -q reportlab
    from reportlab.pdfgen import canvas
    os.makedirs("data/policy", exist_ok=True)
    c = canvas.Canvas("data/policy/compliance_policy.pdf")
    lines = [
        "Section 3.3.2", "Pedestrian Movement", "WARNING",
        "Workers must remain within designated yellow walkway boundaries at all times.",
        "Unsafe behavior includes crossing forklift travel lanes outside marked crosswalks.",
        "",
        "Section 3.4.1", "Equipment Intervention", "WARNING",
        "Personnel must wear high-visibility vests before approaching active machinery.",
        "",
        "Section 4.1.1", "Electrical Panel Management", "CRITICAL SAFETY NOTICE",
        "Panels must remain closed and locked except during authorized maintenance.",
        "An open, unattended panel constitutes a critical hazard.",
        "",
        "Section 4.2.3", "Forklift Load Management", "CRITICAL SAFETY NOTICE",
        "Forklifts must not carry more than two stacked blocks per the rated load chart.",
        "A third stacked block is an immediate overload hazard.",
    ]
    y = 750
    for line in lines:
        c.drawString(50, y, line)
        y -= 20
    c.save()
    print("Synthetic policy PDF written to data/policy/compliance_policy.pdf")
else:
    print("Using existing data/policy/compliance_policy.pdf")

## 6. Build the offline RAG vector store

In [ ]:
import time
t0 = time.time()
!python3 -m src.policy_agent.build_index
print(f"Index build took {time.time() - t0:.1f}s")
!ls -la vector_store/

## 7. Run sample retrieval queries

In [ ]:
import time, sys
sys.path.insert(0, ".")
from src.policy_agent.retriever import get_retriever
from src.policy_agent.rule_reasoner import get_severity_signal

t0 = time.time()
retriever = get_retriever()
print(f"Retriever loaded in {time.time() - t0:.2f}s")

queries = [
    "pedestrian walkway boundary violation",
    "forklift overload safety",
    "electrical panel open unattended",
    "high visibility vest equipment",
]

for q in queries:
    t0 = time.time()
    results = retriever.retrieve(q, k=2)
    dt = time.time() - t0
    print(f"\nQuery: {q!r}  (retrieved in {dt*1000:.1f}ms)")
    for r in results:
        print(f"  [{r.section_ref}] callout={r.callout} score={r.score:.3f}")
        print(f"    {r.text[:120]}...")

print("\n--- Severity signal lookups ---")
for label in ["Pedestrian Movement", "Electrical Panel Management", "Forklift Load Management"]:
    sig = get_severity_signal(label, retriever=retriever)
    print(f"{label}: callout={sig.callout_level}  section={sig.supporting_section_ref}")

## 8. Run the ABM core (Policy → Vision → Risk → Escalation → Audit)

This runs the actual Mesa model over every clip discovered in
`data/raw_clips/`. Vision/Risk/Escalation/Audit are currently stubs, so
this verifies scheduling + blackboard wiring, not detection accuracy.

In [ ]:
import importlib, time
import src.abm.model as abm_model
importlib.reload(abm_model)

t0 = time.time()
model = abm_model.run_batch()
dt = time.time() - t0

print(f"\nABM batch run took {dt:.2f}s")
print("Run stats:", model.run_stats)
print("\nRegistered agents:", list(model.agents_by_role.keys()))
print("Scheduler agent count:", model.schedule.get_agent_count())

## 9. Quick Green-AI resource check

Lightweight, dependency-free timing/memory snapshot — not a substitute for
the full `docs/EVALUATION.md` methodology, but useful for a fast sanity
check that nothing in this pipeline is unexpectedly heavy.

In [ ]:
import resource, platform
peak_rss_mb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024
print(f"Platform: {platform.platform()}")
print(f"Peak resident memory this session: {peak_rss_mb:.1f} MB")
print("No GPU was requested or used at any point in this notebook.")

## 10. Summary

If every cell above ran without error, the following are verified working end-to-end:

- ✅ Repo clones and installs cleanly with CPU-only dependencies
- ✅ Kaggle dataset auto-download (given valid credentials)
- ✅ PDF → section/callout parsing → chunking → local embeddings → FAISS index
- ✅ Retrieval returns policy-grounded, cited passages (section ref + WARNING/CRITICAL callout)
- ✅ Mesa ABM core schedules all five agents and runs the blackboard chain per clip

Next milestone: fill in Vision/Risk/Escalation/Audit agent logic, then re-run
this notebook — step 8's `run_stats` should start showing non-empty
`detections_by_class` / `severities_by_tier` counts.

## 11. Start the FastAPI backend and confirm it's healthy

All five agents (Policy, Vision, Risk, Escalation, Audit) are now fully
implemented. This starts the backend in the background and checks
`/api/health`, confirming the REST + SSE API used by the React dashboard
is reachable. In Colab, the dashboard itself isn't run here (it needs a
local `npm run dev` process and a browser) — this cell only verifies the
backend half of the stack.

In [ ]:
import subprocess, time, requests

backend_proc = subprocess.Popen(
    ["uvicorn", "src.backend.main:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

healthy = False
for attempt in range(15):
    time.sleep(2)
    try:
        r = requests.get("http://localhost:8000/api/health", timeout=2)
        if r.status_code == 200:
            print("Backend healthy:", r.json())
            healthy = True
            break
    except requests.exceptions.ConnectionError:
        print(f"Waiting for backend to start... (attempt {attempt + 1}/15)")

if not healthy:
    print("Backend did not become healthy in time. Recent log output:")
    out, _ = backend_proc.communicate(timeout=2)
    print(out[-2000:])
else:
    print("\nTry it: http://localhost:8000/api/reports")
    print("To run the dashboard locally (not in Colab): cd src/dashboard && npm install && npm run dev")

In [ ]:
# Optional cleanup — stop the backend process when you're done testing.
backend_proc.terminate()
print("Backend stopped.")